In [0]:
# Method: SCD2 initial load
# Objective: build the first version of dim_account, with every account as 'standard'
from pyspark.sql import functions as F

silver = spark.table("workspace.silver.transactions")
start_date = silver.agg(F.min("transaction_date")).first()[0]   # first day in the data

dim_account_scd2 = (
    spark.table("workspace.gold.dim_account")                   # account_key, account_id
    .withColumn("segment", F.lit("standard"))
    .withColumn("valid_from", F.lit(start_date))                # version starts here
    .withColumn("valid_to", F.lit("9999-12-31").cast("date"))   # open-ended = still valid
    .withColumn("is_current", F.lit(True))
    # one key PER VERSION: the same account will have two versions later
    .withColumn("account_version_key",
                F.xxhash64("account_id", F.col("valid_from").cast("string")))
    .select("account_version_key", "account_key", "account_id",
            "segment", "valid_from", "valid_to", "is_current")
)

(dim_account_scd2.write.format("delta").mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("workspace.gold.dim_account_scd2"))

display(dim_account_scd2)

In [0]:
%sql
-- Method: COUNT + FILTER
-- Objective: how many account versions exist and how many are current standard?
SELECT COUNT(*) AS total_rows,
       SUM(CASE WHEN is_current AND segment = 'standard' THEN 1 ELSE 0 END) AS current_standard
FROM workspace.gold.dim_account_scd2

In [0]:
# Method: pick one account (always the same one)
# Objective: choose the account that will change from 'standard' to 'premium'
account_id = (spark.table("workspace.gold.dim_account_scd2")
              .orderBy("account_id").first()["account_id"])
print(account_id)

In [0]:
# Method: SCD2 change = UPDATE (close old version) + INSERT (open new version)
# Objective: the account becomes 'premium' on 2026-10-01, keeping the 'standard' history
# Run this cell ONCE (if you run it twice, rerun step 11.1 to reset the table)

# 1) Close the current version: it is valid until the day before the change
spark.sql(f"""
    UPDATE workspace.gold.dim_account_scd2
    SET valid_to = DATE'2026-09-30', is_current = FALSE
    WHERE account_id = '{account_id}' AND is_current = TRUE
""")

# 2) Open the new version: same account, new segment, valid from the change date
spark.sql(f"""
    INSERT INTO workspace.gold.dim_account_scd2
    SELECT xxhash64(account_id, '2026-10-01'),      -- new version key
           account_key, account_id,
           'premium', DATE'2026-10-01', DATE'9999-12-31', TRUE
    FROM workspace.gold.dim_account
    WHERE account_id = '{account_id}'
""")

In [0]:
# Method: filter + order
# Objective: show the two versions of the changed account and the table totals
display(spark.table("workspace.gold.dim_account_scd2")
        .filter(f"account_id = '{account_id}'").orderBy("valid_from"))

print(spark.table("workspace.gold.dim_account_scd2").count(),
      spark.table("workspace.gold.dim_account_scd2").filter("is_current").count())

In [0]:
%sql
-- Method: JOIN with a date range (point-in-time join)
-- Objective: which segment did account ACC001 have on the day of each transaction?
SELECT f.transaction_id, d.full_date, a.segment, f.amount, f.status
FROM workspace.gold.fact_transactions f
JOIN workspace.gold.dim_date d
  ON f.date_key = d.date_key
JOIN workspace.gold.dim_account_scd2 a
  ON f.account_key = a.account_key                         -- same account
 AND d.full_date BETWEEN a.valid_from AND a.valid_to       -- version valid on that day
WHERE a.account_id = 'ACC001'
ORDER BY d.full_date

In [0]:
%sql
-- Method: row count after the join vs distinct transactions
-- Objective: does every transaction match exactly ONE account version?
SELECT COUNT(*) AS rows_after_join,
       COUNT(DISTINCT f.transaction_id) AS distinct_transactions
FROM workspace.gold.fact_transactions f
JOIN workspace.gold.dim_date d ON f.date_key = d.date_key
JOIN workspace.gold.dim_account_scd2 a
  ON f.account_key = a.account_key
 AND d.full_date BETWEEN a.valid_from AND a.valid_to